<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/%5B03D%5D-Item_Code_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3d: 8-K Item-Code Filter (Objective)

Every 8-K filing declares which numbered "Item" it's reporting under, right
on its cover page — this is metadata the filer checked off, not something
requiring interpretation. Your Synopsis explicitly scopes the study to
**Items 8.01, 1.01, and 2.01 only**.

This notebook pulls each filing's declared item code(s) from SEC's own
submissions API and auto-excludes anything filed under an out-of-scope item
(most commonly Item 2.02 — Results of Operations, i.e. an earnings release
that happened to mention AI in passing, not a genuine investment
announcement). This is purely a metadata check, not a content judgment, so
it's fair game to automate.

**Run this after `03c_automated_flags.ipynb`.**

**v2 note:** this version fixes a bug from the first release where
`NaN or ""` evaluated to `NaN` (Python treats NaN as truthy) instead of
`""`, which corrupted every clean row's `exclude_reason` into the literal
text `"nan"` and made the summary count everything as excluded. Cell 4b
below repairs that if it already happened to your data.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "your_email@example.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 247 (delta 7), reused 8 (delta 2), pack-reused 228 (from 1)
Receiving objects: 100% (247/247), 2.01 MiB | 4.13 MiB/s, done.
Resolving deltas: 100% (104/104), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas requests

## Run logging (for the pipeline job-history dashboard)

Logs this notebook's start time to `results/run_log.csv` and installs an
error hook that automatically records which cell failed, if anything does.
This runs once per session and needs no attention - just let it run.

In [ ]:
import time, uuid, subprocess
import pandas as pd
from datetime import datetime, timezone

NOTEBOOK_NAME = "03d_item_code_filter"
RUN_LOG = os.path.join(BASE_DIR, "results", "run_log.csv")
os.makedirs(os.path.dirname(RUN_LOG), exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
RUN_START = time.time()
RUN_LOG_COLS = ["run_id", "notebook", "status", "start_time", "end_time",
                "duration_seconds", "failed_stage", "error_message"]

def _log_row(status, stage="", error=""):
    row = {
        "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "status": status,
        "start_time": datetime.fromtimestamp(RUN_START, tz=timezone.utc).isoformat(),
        "end_time": datetime.now(timezone.utc).isoformat() if status != "running" else "",
        "duration_seconds": round(time.time() - RUN_START, 1) if status != "running" else "",
        "failed_stage": stage, "error_message": str(error)[:300],
    }
    if os.path.exists(RUN_LOG):
        log_df = pd.read_csv(RUN_LOG)
        log_df = log_df[log_df["run_id"] != RUN_ID]
    else:
        log_df = pd.DataFrame(columns=RUN_LOG_COLS)
    log_df = pd.concat([log_df, pd.DataFrame([row])], ignore_index=True)
    log_df.to_csv(RUN_LOG, index=False)

def _push_log(message):
    subprocess.run(["git", "-C", BASE_DIR, "add", "results/run_log.csv"], capture_output=True)
    subprocess.run(["git", "-C", BASE_DIR, "commit", "-m", message], capture_output=True)
    subprocess.run(["git", "-C", BASE_DIR, "push"], capture_output=True)

_log_row("running")
_push_log(f"Run log: {NOTEBOOK_NAME} started")

def _custom_exc(shell, etype, evalue, tb, tb_offset=None):
    stage = f"In [{shell.execution_count}]"
    _log_row("failed", stage=stage, error=evalue)
    _push_log(f"Run log: {NOTEBOOK_NAME} FAILED at {stage}")
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)

get_ipython().set_custom_exc((Exception,), _custom_exc)
print(f"Run logging active for {NOTEBOOK_NAME} (run_id={RUN_ID})")

Run logging active for 03d_item_code_filter (run_id=20260726T132349-9c23fe)


## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
HEADERS = {"User-Agent": "QM640 Capstone research your_email@example.com"}  # edit to your real email

IN_SCOPE_ITEMS = {"1.01", "2.01", "8.01"}  # per Synopsis Scope Boundaries

## Cell 4 — Load worksheet

In [ ]:
import pandas as pd

df = pd.read_csv(SCREENING_FILE)
print(f"Loaded {len(df)} rows")
still_open = df["exclude_reason"].isna() | (df["exclude_reason"].astype(str).str.strip() == "")
print(f"Rows not yet auto-excluded going into this notebook: {still_open.sum()}")

Loaded 9631 rows
Rows not yet auto-excluded going into this notebook: 1797


## Cell 4b — Repair pass (safe to run even if you haven't hit this bug)

An earlier version of this notebook had a bug: `NaN or ""` evaluates to
`NaN` in Python (NaN is truthy, not falsy), not `""`. That turned every
genuinely clean row's blank `exclude_reason` into the literal text `"nan"`,
which then made every row look excluded downstream. This cell repairs that
corruption if present; it's a no-op if your data is already clean.

In [ ]:
def clean_reason(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x == "nan":
        return ""
    if x.lower().startswith("nan; "):
        return x[5:]
    return x

before = (df["exclude_reason"].astype(str).str.strip() == "nan").sum()
df["exclude_reason"] = df["exclude_reason"].apply(clean_reason)
print(f"Repaired {before} rows that had the literal string artifact from the old bug")

still_open = df["exclude_reason"].astype(str).str.strip() == ""
print(f"Rows genuinely open (no exclusion reason) after repair: {still_open.sum()}")

Repaired 1797 rows that had the literal string artifact from the old bug
Rows genuinely open (no exclusion reason) after repair: 1797


## Cell 5 — Pull declared item codes per filing (SEC submissions API)

Caches one call per unique CIK, matches by accession number to find each
row's own filing within that company's recent filing history.

In [ ]:
import requests
import time

def get_filing_history_with_items(cik):
    cik_padded = str(int(cik)).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    if resp.status_code != 200:
        return None
    data = resp.json()
    recent = data.get("filings", {}).get("recent", {})
    return pd.DataFrame({
        "accessionNumber": recent.get("accessionNumber", []),
        "items": recent.get("items", []),
    })


if "item_codes" not in df.columns:
    df["item_codes"] = None

already_done = df["item_codes"].notna() & (df["item_codes"].astype(str).str.strip() != "")
to_process = df[~already_done]
print(f"{already_done.sum()} rows already have item codes from a prior run - skipping")
print(f"{len(to_process)} rows need processing")

history_cache = {}
new_item_codes = {}

for idx, row in to_process.iterrows():
    cik = row["cik"]
    if cik not in history_cache:
        history_cache[cik] = get_filing_history_with_items(cik)
        time.sleep(0.15)

    hist = history_cache[cik]
    own_accn = row["accession_no"].split(":")[0] if isinstance(row["accession_no"], str) else None

    if hist is None or hist.empty or own_accn is None:
        new_item_codes[idx] = None
        continue

    match = hist[hist["accessionNumber"] == own_accn]
    new_item_codes[idx] = match["items"].iloc[0] if not match.empty else None

for idx, code in new_item_codes.items():
    df.loc[idx, "item_codes"] = code

print(df["item_codes"].value_counts().head(20))

0 rows already have item codes from a prior run - skipping
9631 rows need processing
item_codes
2.02,9.01              2580
7.01,9.01              1530
8.01,9.01              1007
2.02,7.01,9.01          985
1.01,7.01,9.01          296
5.02,7.01,9.01          274
2.02,8.01,9.01          243
7.01,8.01,9.01          221
1.01,8.01,9.01          162
5.02,9.01               126
1.01,3.02,7.01,9.01     110
1.01,9.01               108
2.02,5.02,9.01           87
2.02,7.01,8.01,9.01      75
1.01,3.02,8.01,9.01      75
9.01                     61
7.01                     55
5.02,8.01,9.01           55
8.01                     46
1.01,7.01,8.01,9.01      46
Name: count, dtype: int64


## Cell 6 — Flag out-of-scope items and update exclude_reason

Fixed: NaN is now handled explicitly with `pd.isna()` instead of the buggy
`x or default` pattern.

In [ ]:
def is_in_scope(items_str):
    if pd.isna(items_str) or items_str == "":
        return "REVIEW"  # could not determine - needs a manual glance
    filed_items = {i.strip() for i in str(items_str).split(",")}
    return "Y" if filed_items & IN_SCOPE_ITEMS else "N"


df["item_in_scope"] = df["item_codes"].apply(is_in_scope)


def update_reason(row):
    raw = row.get("exclude_reason", "")
    existing = "" if pd.isna(raw) else str(raw).strip()

    if row["item_in_scope"] == "N":
        addition = f"filed under out-of-scope item(s): {row['item_codes']}"
        return f"{existing}; {addition}" if existing else addition
    return existing


df["exclude_reason"] = df.apply(update_reason, axis=1)
df.to_csv(SCREENING_FILE, index=False)

print(df["item_in_scope"].value_counts())
print(f"\nSaved -> {SCREENING_FILE}")

item_in_scope
N         6119
Y         3451
REVIEW      61
Name: count, dtype: int64

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3d: 8-K item-code scope filter (v2, with nan-string bug fix)"
!git -C {BASE_DIR} push

[main 0d08794] Step 3d: 8-K item-code scope filter (v2, with nan-string bug fix)
 1 file changed, 9632 insertions(+), 9632 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 153.81 KiB | 1.67 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   4a9ef25..0d08794  main -> main


## Summary — updated manual-review workload

In [ ]:
def is_excluded(x):
    return isinstance(x, str) and len(x.strip()) > 0

auto_excluded = df["exclude_reason"].apply(is_excluded)
needs_review = df["item_in_scope"] == "REVIEW"

print(f"Total candidate filings: {len(df)}")
print(f"Auto-excluded so far (Step 3c + 3d combined): {auto_excluded.sum()}")
print(f"Item code lookup failed, needs a quick manual glance: {needs_review.sum()}")
print(f"Remaining for real manual content review (is_genuine_ai_event / announcement_type): "
      f"{(~auto_excluded).sum()}")

Total candidate filings: 9631
Auto-excluded so far (Step 3c + 3d combined): 8889
Item code lookup failed, needs a quick manual glance: 61
Remaining for real manual content review (is_genuine_ai_event / announcement_type): 742


## Export a small, focused file for the actual manual review

Instead of scrolling through all 2,000+ rows in the master worksheet, this
pulls out just the ones that still need your eyes -- the genuinely open rows
plus the item-code lookup failures -- into a separate, much smaller CSV.

You'll still edit the *master* `screening_worksheet.csv` (matched by
`accession_no`), but this file is what you actually open and work through.

In [ ]:
to_review = df[~auto_excluded | needs_review].copy()
review_cols = ["accession_no", "company_name", "file_date", "announcement_type",
               "is_genuine_ai_event", "item_codes", "item_in_scope", "filing_url"]
to_review = to_review[review_cols]

review_path = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")
to_review.to_csv(review_path, index=False)
print(f"{len(to_review)} rows to review -> {review_path}")
print("\nWorkflow: fill in is_genuine_ai_event / announcement_type in THIS file, "
      "then merge those two columns back into screening_worksheet.csv by accession_no "
      "before running the kappa check (Part B of notebook 03).")

769 rows to review -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_TO_REVIEW.csv

Workflow: fill in is_genuine_ai_event / announcement_type in THIS file, then merge those two columns back into screening_worksheet.csv by accession_no before running the kappa check (Part B of notebook 03).


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_TO_REVIEW.csv"
!git -C {BASE_DIR} commit -m "Step 3d: export focused manual-review subset"
!git -C {BASE_DIR} push

[main 80254c0] Step 3d: export focused manual-review subset
 1 file changed, 770 insertions(+), 246 deletions(-)
 rewrite data/raw/screening_TO_REVIEW.csv (94%)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 23.17 KiB | 2.57 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   0d08794..80254c0  main -> main


## Mark this run complete (for the job-history log)

In [ ]:
_log_row("success")
_push_log(f"Run log: {NOTEBOOK_NAME} completed successfully")
print(f"Run logged as SUCCESS. Duration: {round(time.time() - RUN_START, 1)}s")

Run logged as SUCCESS. Duration: 564.5s
